# Glasser RSA boilerplate

In [1]:
import os
import numpy as np
from scipy.stats import gamma
import subprocess
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr, kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")

In [2]:
def preprocess_fmri(fmri_data, timing_df, TR, bin_sec, delay_sec, drop_last=True):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    """
    n_vertices, n_timepoints = fmri_data.shape
    segments = []
    
    for _, row in timing_df.iterrows():
        # --- FIX: Removed trailing commas below ---
        onset_sec = row["onset_sec"]
        duration_sec = row["duration_sec"]

        # Apply delay to the stimulus window
        onset_d = onset_sec + delay_sec
        end_d = onset_sec + duration_sec + delay_sec

        # Convert to timepoint indices
        start_tp = int(np.floor(onset_d / TR)) 
        end_tp = int(np.ceil(end_d / TR))

        # Clip to valid range
        start_tp = max(start_tp, 0)
        end_tp = min(end_tp, n_timepoints)

        # Handle edge case where delay pushes segment out of bounds
        if end_tp <= start_tp:
            raise RuntimeError(f'Segmentation error from {onset_sec} to {onset_sec+duration_sec}') 

        segment = fmri_data[:, start_tp:end_tp]

        # Timepoints per bin
        tp_per_bin = int(np.floor(bin_sec / TR))
        if tp_per_bin < 1:
            raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

        # Drop leftover timepoints that don't fill a full bin
        if segment.shape[1] == 0:
            continue
            
        if drop_last:
            n_bins = int(np.floor(segment.shape[1] / tp_per_bin))
        else:
            n_bins = int(np.ceil(segment.shape[1] / tp_per_bin))
            
        # Ensure we have enough data for at least one bin
        if n_bins == 0:
            continue
            
        # Truncate to exact multiple of bin size to allow reshape
        # If not dropping last, we might need padding, but standard approach here 
        # usually implies we only take what fits or we strictly reshape.
        # Given your code used reshape(), we must ensure exact size.
        valid_length = n_bins * tp_per_bin
        
        # If strict floor was used, this just trims tail
        if segment.shape[1] >= valid_length:
            segment = segment[:, :valid_length]
        else:
            # If we calculated bins via ceil but don't have enough data (rare), skip
            continue

        binned = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
        segments.append(binned)
        # print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    if len(segments) == 0:
        return np.zeros((n_vertices, 0), dtype=fmri_data.dtype)
        
    filtered_fmri = np.concatenate(segments, axis=1)
    return filtered_fmri

In [3]:
def load_glasser_parcellation(label_file_left, label_file_right):
    """
    Load Glasser parcellation from separate GIFTI label files.
    
    Parameters:
    -----------
    label_file_left : str
        Path to left hemisphere .label.gii file
    label_file_right : str
        Path to right hemisphere .label.gii file
    
    Returns:
    --------
    left_labels : np.ndarray
        Left hemisphere parcel IDs (n_vertices,)
    right_labels : np.ndarray
        Right hemisphere parcel IDs (n_vertices,)
    left_info : dict
        Left hemisphere parcel info {id: name}
    right_info : dict
        Right hemisphere parcel info {id: name}
    """
    print(f"Loading Glasser parcellation from separate hemisphere files...")
    
    # Load left hemisphere
    print(f"\nLeft hemisphere: {label_file_left}")
    gii_left = nib.load(label_file_left)
    left_labels = gii_left.darrays[0].data.astype(int)
    
    # Extract label names from label table
    left_info = {}
    if hasattr(gii_left, 'labeltable') and gii_left.labeltable is not None:
        # labeltable.labels is a LIST of GiftiLabel objects
        for label_obj in gii_left.labeltable.labels:
            key = label_obj.key
            label_name = label_obj.label
            if key > 0:  # Skip background (0)
                left_info[key] = label_name
    
    # Load right hemisphere
    print(f"Right hemisphere: {label_file_right}")
    gii_right = nib.load(label_file_right)
    right_labels = gii_right.darrays[0].data.astype(int)
    
    # Extract label names from label table
    right_info = {}
    if hasattr(gii_right, 'labeltable') and gii_right.labeltable is not None:
        # labeltable.labels is a LIST of GiftiLabel objects
        for label_obj in gii_right.labeltable.labels:
            key = label_obj.key
            label_name = label_obj.label
            if key > 0:  # Skip background (0)
                right_info[key] = label_name
    
    # Get statistics
    left_unique = np.unique(left_labels)
    left_unique = left_unique[left_unique > 0]
    right_unique = np.unique(right_labels)
    right_unique = right_unique[right_unique > 0]
    
    print(f"\nLeft hemisphere: {len(left_unique)} parcels")
    print(f"  Parcel IDs range: {left_unique.min()} to {left_unique.max()}")
    print(f"  Total vertices: {len(left_labels)}")
    
    print(f"\nRight hemisphere: {len(right_unique)} parcels")
    print(f"  Parcel IDs range: {right_unique.min()} to {right_unique.max()}")
    print(f"  Total vertices: {len(right_labels)}")
    
    return left_labels, right_labels, left_info, right_info

In [4]:
def spm_hrf(tr, oversampling=16):
    dt = tr / oversampling
    u = np.arange(0, 32, dt) - dt/2
    h = (6 * gamma.pdf(u, 6, loc=0, scale=1) - 
         1/6 * gamma.pdf(u, 16, loc=0, scale=1))
    return h / np.sum(h)



def process_model_embeddings(model_embeddings, bin_sec, timing_df, hrf=True, normalization=False):
    """
    Handles HRF convolution and PER-RUN normalization/demeaning.
    """
    # 1. Setup HRF
    hrf_kernel = spm_hrf(bin_sec, oversampling=1)
    
    # 2. Slice into individual video segments
    video_bin_counts = (timing_df['duration_sec'] / bin_sec).astype(int).values
    segments = []
    current_idx = 0
    
    for n_bins in video_bin_counts:
        segment = model_embeddings[current_idx : current_idx + n_bins, :]
        
        # Apply HRF if requested
        if hrf:
            conv_segment = np.zeros_like(segment)
            for i in range(segment.shape[1]):
                conv = np.convolve(segment[:, i], hrf_kernel, mode='full')
                conv_segment[:, i] = conv[:n_bins]
            segments.append(conv_segment)
        else:
            segments.append(segment)
            
        current_idx += n_bins

    # 3. Perform Per-Run Normalization OR Demeaning
    print(f"Performing Per-Run {'Normalization (Z-score)' if normalization else 'Demeaning (Mean centering)'}...")
    normalized_segments = [None] * len(segments)
    unique_runs = timing_df['run_id'].unique()
    
    for rid in unique_runs:
        run_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
        run_data = np.concatenate([segments[i] for i in run_indices], axis=0)
        
        if normalization:
            # Z-score the entire run block
            run_proc = zscore(run_data, axis=0)
            run_proc = np.nan_to_num(run_proc) # Clean up constant features
        else:
            # Demean per run (Subtract the mean of the run from the run data)
            run_proc = run_data - np.mean(run_data, axis=0)
        
        # Slice the processed run block back into original segments
        slice_ptr = 0
        for i in run_indices:
            v_len = video_bin_counts[i]
            normalized_segments[i] = run_proc[slice_ptr : slice_ptr + v_len, :]
            slice_ptr += v_len
    
    segments = normalized_segments

    # 4. Re-concatenate into final embedding matrix
    return np.concatenate(segments, axis=0)


In [5]:
def compute_rsa_map(fmri_data, timing_df, bin_sec, parcel_labels, parcel_info, model_data, output_path,
                    normalization=False, hrf=False, method='spearman',
                    save_pvalues=True, min_variance=1e-10, 
                    block_diagonal=False):
    """
    Compute RSA correlation map for all Glasser parcels.
    """
    n_vertices, n_timepoints = fmri_data.shape
    print(f"fMRI data shape: {n_vertices} vertices × {n_timepoints} timepoints")
    
    # -------------------------------------------------------------------------
    # 1. LOAD AND PREPROCESS MODEL
    # -------------------------------------------------------------------------
    if isinstance(model_data, (str, Path)):
        raw_model = np.load(model_data)
    else:
        raw_model = model_data

    clean_model = process_model_embeddings(
        raw_model, 
        bin_sec=bin_sec, 
        timing_df=timing_df, 
        hrf=hrf, 
        normalization=normalization
    )

    # Verify model shape matches fMRI time
    if clean_model.shape[0] != n_timepoints:
        raise ValueError(
            f"Model timepoints ({clean_model.shape[0]}) doesn't match "
            f"fMRI timepoints ({n_timepoints})"
        )

    # -------------------------------------------------------------------------
    # 2. PREPARE MODEL VECTOR (Global vs Block-Diagonal)
    # -------------------------------------------------------------------------
    
    # Pre-calculate video segment indices
    video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values
    limits = []
    current_idx = 0
    for b in video_bins:
        limits.append((current_idx, current_idx + b))
        current_idx += b

    if block_diagonal:
        print("\n--- STRATEGY: BLOCK-DIAGONAL (WITHIN-VIDEO) RSA ---")
        model_vecs = []
        valid_indices_mask = [] # To track which segments are valid (len > 2)
        
        for i, (start, end) in enumerate(limits):
            vid_model = clean_model[start:end, :]
            
            # Skip short segments (need at least 3 points for meaningful correlation distance)
            if vid_model.shape[0] < 3: 
                valid_indices_mask.append(False)
                continue
            
            valid_indices_mask.append(True)
            model_vecs.append(pdist(vid_model, metric='correlation'))
            
        final_model_vec = np.hstack(model_vecs)
        
    else:
        print("\n--- STRATEGY: GLOBAL RSA ---")
        final_model_vec = pdist(clean_model, metric='correlation')

    print(f"Final Model Vector elements: {len(final_model_vec)}")
    
    if final_model_vec.std() < min_variance:
        raise ValueError("Model RDM has zero variance! Cannot compute correlations.")

    # -------------------------------------------------------------------------
    # 3. COMPUTE RSA FOR PARCELS
    # -------------------------------------------------------------------------
    
    unique_parcels = np.unique(parcel_labels)
    unique_parcels = unique_parcels[unique_parcels > 0]
    
    print(f"\nComputing RSA for {len(unique_parcels)} parcels...")
    
    rsa_correlations = np.zeros(n_vertices, dtype=np.float32)
    if save_pvalues:
        rsa_pvalues = np.ones(n_vertices, dtype=np.float32)
    
    n_computed = 0
    n_skipped = 0
    n_failed = 0
    parcel_results_list = []
    
    for i, parcel_id in enumerate(unique_parcels):
        if (i + 1) % 50 == 0:
            print(f"Progress: {i+1}/{len(unique_parcels)}", end = "\r")
            print("")
            
        try:
            parcel_mask = parcel_labels == parcel_id
            if parcel_mask.sum() == 0: 
                n_skipped += 1
                continue
            
            parcel_activations = fmri_data[parcel_mask, :].T 
            
            if parcel_activations.std() < min_variance:
                n_skipped += 1
                continue
                
            if block_diagonal:
                brain_vecs = []
                for idx, (start, end) in enumerate(limits):
                    if not valid_indices_mask[idx]: 
                        continue
                    vid_brain = parcel_activations[start:end, :]
                    brain_vecs.append(pdist(vid_brain, metric='correlation'))
                
                if len(brain_vecs) == 0:
                    n_skipped += 1
                    continue
                final_brain_vec = np.hstack(brain_vecs)
            else:
                final_brain_vec = pdist(parcel_activations, metric='correlation')
            
            if len(final_brain_vec) != len(final_model_vec):
                n_failed += 1
                continue
                
            if method == 'spearman':
                corr, pval = spearmanr(final_brain_vec, final_model_vec, nan_policy='omit')
            elif method == 'pearson':
                corr, pval = pearsonr(final_brain_vec, final_model_vec)
            elif method == 'kendall':
                corr, pval = kendalltau(final_brain_vec, final_model_vec, nan_policy='omit')
            
            if np.isnan(corr):
                n_failed += 1
                continue
                
            rsa_correlations[parcel_mask] = corr
            if save_pvalues:
                rsa_pvalues[parcel_mask] = pval
                
            parcel_name = parcel_info.get(parcel_id, f"Parcel_{parcel_id}")
            parcel_results_list.append({
                'parcel_id': parcel_id,
                'parcel_name': parcel_name,
                'correlation': corr,
                'pvalue': pval
            })
            n_computed += 1

        except Exception:
            n_failed += 1
            continue

    # -------------------------------------------------------------------------
    # 4. SAVE AND REPORT
    # -------------------------------------------------------------------------
    print(f"\nRSA computation complete! Computed: {n_computed}, Skipped: {n_skipped}, Failed: {n_failed}")
    
    if output_path:
        output_path = Path(output_path)
        np.save(output_path.with_suffix('.npy'), rsa_correlations)
        if save_pvalues:
            np.save(output_path.with_name(output_path.stem + '_pvalues.npy'), rsa_pvalues)
            
    return rsa_correlations

In [6]:
# ============================================================================
# COMPLETE PIPELINE FUNCTION
# ============================================================================
def complete_rsa_pipeline(hemisphere, label, parcel_labels, parcel_info, fmri_data, 
                          timing_df,bin_sec, model_data, output_dir, 
                          normalization, hrf, method, 
                          block_diagonal):
    """
    Complete Glasser parcellation RSA pipeline.
    
    Parameters:
    -----------
    hemisphere: str
        'left' or 'right'.
    label : str
        Label for the file naming (e.g. 'pe-av-small-16-frame_video').
    parcel_labels : np.ndarray
        Parcel labels for each vertex.
    parcel_info : dict
        Parcel name dictionary.
    fmri_data : np.ndarray
        Preprocessed fMRI data (n_vertices, n_timepoints).
    timing_df: pd.DataFrame
        Timing informaton for each video
    bin_sec : float
        Bin size in seconds ( equal to TR or multiple).
    model_data : str or np.ndarray
        Path to model embeddings (.npy file) or loaded array.
    output_dir : str
        Output directory path.
    normalization : bool
        Whether to z-score the model.
    hrf : bool
        Whether to apply HRF convolution.
    method : str
        'spearman', 'pearson', or 'kendall'.
    block_diagonal : bool
        If True, performs Within-Video RSA (concatenating blocks).
        If False, performs Standard Global RSA.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    mode_str = "BLOCK-DIAGONAL (Within-Video)" if block_diagonal else "GLOBAL"
    
    print("=" * 70)
    print(f"GLASSER PARCELLATION RSA PIPELINE [{mode_str}]")
    print("=" * 70)
    
    # Compute RSA map
    print(f"\nSTEP 1: Computing RSA correlation map ({hemisphere})...")
    print("-" * 70)
    
    # Add suffix to filename
    suffix = "blockdiag" if block_diagonal else "global"
    rsa_output = output_dir / f"rsa_{label}_glasser_{method}_{'normalized' if normalization else 'notnormalized'}_{'hrf' if hrf else 'nohrf'}_bin{int(bin_sec)}s_{hemisphere}_{suffix}"
    
    rsa_correlations = compute_rsa_map(
        fmri_data=fmri_data,
        timing_df=timing_df,
        bin_sec =bin_sec,
        parcel_labels=parcel_labels,
        parcel_info=parcel_info,
        model_data=model_data,
        output_path=str(rsa_output),
        normalization=normalization,  
        hrf=hrf,                      
        method=method,
        save_pvalues=True,
        min_variance=1e-10,
        block_diagonal=block_diagonal 
    )
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE!")
    print("=" * 70)
    print(f"Non-Image Results saved in: {output_dir}")
    
    return rsa_correlations

# automated mega loop

In [8]:
import numpy as np
import pandas as pd
import scipy.io as sio
from pathlib import Path
import sys
from scipy.stats import zscore

# ============================================================================
# CONFIGURATION
# ============================================================================

''' 
Based on viewing the results, this configuration was chosen. de-meaning was added to when normalization is not performed. as a result the un-normalized results 
of configs other than 5s delay which didnt use normalization are not the latest version of the code 
(previous version used nothing when normalization is false; current one does demeaning). so the the only cofigurations with the latest variations in code are the ones
who are saved as cifti.
'''
use_normalization = [True , False]
use_hrf = [True , False]
BLOCKDIAG = [True, False]
BIN_SEC_LIST = [1.0, 2.0, 5.0, 10.0]  
DELAY_SEC_LIST = [5.0]

# Paths
template_cifti_path = '/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii'
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")
MAT_FILE_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
MAT_FILE_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"
GLASSER_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
GLASSER_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"
TIMING_CSV = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"

# Models & Modalities
models = {"pe-av-small-16-frame"}
modalities = {
    "joint": "av",
    # "audio": "a",
    # "video": "v"
}
RSA_METHODS = ['spearman']
TR = 1.0  

# ============================================================================
# STEP 0: LOAD RAW DATA ONCE
# ============================================================================
print("=" * 70)
print("LOADING RAW DATA (Running Once)")
print("=" * 70)

timing_df = pd.read_csv(TIMING_CSV)

# Load Glasser Labels (Assumes function is defined in your environment)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(GLASSER_LEFT, GLASSER_RIGHT)

# Load fMRI data
print("Loading Hemispheres...")
fmri_left_raw = sio.loadmat(MAT_FILE_LEFT)['Left_data']
fmri_right_raw = sio.loadmat(MAT_FILE_RIGHT)['Right_data']

# Ensure shape is (Vertices, Time)
if fmri_left_raw.shape[0] != 32492: fmri_left_raw = fmri_left_raw.T
if fmri_right_raw.shape[0] != 32492: fmri_right_raw = fmri_right_raw.T

# Pre-calculate valid masks
valid_mask_left = fmri_left_raw.std(axis=1) > 0
valid_mask_right = fmri_right_raw.std(axis=1) > 0

print(f"Data Loaded. Left Valid: {np.sum(valid_mask_left)}, Right Valid: {np.sum(valid_mask_right)}")

# ============================================================================
# MAIN LOOP
# ============================================================================

for hrf in use_hrf:
    # If HRF is ON, Delay is irrelevant (set to 0). 
    current_delays = [0] if hrf else DELAY_SEC_LIST
    
    for delay_sec in current_delays:
        for bin_sec in BIN_SEC_LIST:
            
            # --- EFFICIENCY FIX: PREPROCESS fMRI ONCE PER TIMESCALE/DELAY ---
            print(f"\n[FMRI] Preprocessing: Bin={bin_sec}s, Delay={delay_sec}s, HRF={hrf}")
            
            # 1. LOAD RAW PREPROCESSED DATA (Base Copy)
            fmri_l_base = preprocess_fmri(fmri_left_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)
            fmri_r_base = preprocess_fmri(fmri_right_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)

            unique_runs = timing_df['run_id'].unique()
            video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values

            # --------------------------------------------------------
            # INNER LOOPS: MODEL VARIATIONS
            # --------------------------------------------------------
            for normalization in use_normalization:
                
               # 2. CREATE WORKING COPIES
                fmri_l_proc = fmri_l_base.copy()
                fmri_r_proc = fmri_r_base.copy()
                
                # 3. PERFORM PER-RUN NORMALIZATION OR DEMEANING
                curr_bin_idx = 0 
                for rid in unique_runs:
                    v_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
                    run_bin_length = sum(video_bins[v_indices])
                    run_slice = slice(curr_bin_idx, curr_bin_idx + run_bin_length)
                    
                    if normalization:
                        fmri_l_proc[valid_mask_left, run_slice] = zscore(fmri_l_proc[valid_mask_left, run_slice], axis=1)
                        fmri_r_proc[valid_mask_right, run_slice] = zscore(fmri_r_proc[valid_mask_right, run_slice], axis=1)
                    else:
                        fmri_l_proc[valid_mask_left, run_slice] -= fmri_l_proc[valid_mask_left, run_slice].mean(axis=1, keepdims=True)
                        fmri_r_proc[valid_mask_right, run_slice] -= fmri_r_proc[valid_mask_right, run_slice].mean(axis=1, keepdims=True)
                    
                    curr_bin_idx += run_bin_length
                fmri_l_proc = np.nan_to_num(fmri_l_proc)
                fmri_r_proc = np.nan_to_num(fmri_r_proc)
                

                # 4. RUN RSA LOOP (Block vs Global)
                for blockdiag in BLOCKDIAG:
                    
                    base_lbl = f"{'normalized' if normalization else 'notnormalized'}{'_hrf' if hrf else ''}_{'blockdiag' if blockdiag else 'global'}"
                    if not hrf:
                        base_lbl += f"_delay{int(delay_sec)}s"
                    
                    for model_label in models:
                        for mod_name, mod_suffix in modalities.items():
                            
                            current_run_label = f"{model_label}_{mod_name}"
                            file_prefix = model_label.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
                            
                            # Embeddings path
                            model_data_path = EMBEDDINGS_BASE / model_label / f"{int(bin_sec)}s" / f"{file_prefix}_{mod_suffix}.npy"
                            output_dir = RSA_RESULTS_BASE / model_label / base_lbl / f"{int(bin_sec)}s" / mod_name
                            
                            if not model_data_path.exists():
                                print(f"  [SKIP] Model missing: {model_data_path}")
                                continue

                            print(f"  -> RSA: {model_label} [{mod_name}] | {base_lbl}")
                            
                            for rsa_method in RSA_METHODS:
                                # Process Left Hemisphere
                                left_corr = complete_rsa_pipeline(
                                    hemisphere='left',
                                    label=current_run_label,
                                    parcel_labels=left_labels,
                                    parcel_info=left_info,
                                    fmri_data=fmri_l_proc, # Already normalized per-run
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "left_hemisphere"),
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag
                                )

                                # Process Right Hemisphere
                                right_corr = complete_rsa_pipeline(
                                    hemisphere='right',
                                    label=current_run_label,
                                    parcel_labels=right_labels,
                                    parcel_info=right_info,
                                    fmri_data=fmri_r_proc, # Already normalized per-run
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "right_hemisphere"),
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag
                                )
                                print(f"\n  [CIFTI] Generating direct CIFTI using template header...")
                                    
                                # Use the correlation arrays and clean NaNs
                                left_clean = np.nan_to_num(left_corr, nan=0.0)
                                right_clean = np.nan_to_num(right_corr, nan=0.0)
                                
                                # DROP MEDIAL WALL BEFORE SAVING CIFTI
                                left_masked = left_clean[valid_mask_left]
                                right_masked = right_clean[valid_mask_right]
                                
                                # Combine the masked data (Now perfectly matches 59412 template)
                                combined_data = np.hstack((left_masked, right_masked)).reshape(1, -1)
                                
                                template_img = nib.load(template_cifti_path)
                                
                                if combined_data.shape[1] != template_img.shape[1]:
                                    raise ValueError(
                                        f"Shape mismatch! Concatenated data: {combined_data.shape[1]} vertices, "
                                        f"Template: {template_img.shape[1]}."
                                    )

                                cifti = nib.Cifti2Image(
                                    combined_data.astype(np.float32), 
                                    header=template_img.header,
                                    nifti_header=template_img.nifti_header
                                )
                                
                                # Define and update metric name
                                metric_name = f"RSA_glasser_{rsa_method}_{'blockdiag' if blockdiag else 'global'}"
                                try:
                                    scalar_axis = cifti.header.get_axis(0)
                                    scalar_axis.name[0] = metric_name
                                except Exception as e:
                                    print(f"    (Note: Could not update internal map name: {e})")

                                # Fix Output Path (Create dir + add filename)
                                cifti_dir = RSA_RESULTS_BASE / 'cifti_maps'
                                cifti_dir.mkdir(parents=True, exist_ok=True)
                                map_output = cifti_dir / f"rsa_{current_run_label}_glasser_{base_lbl}_{rsa_method}_{int(bin_sec)}s.dscalar.nii"

                                nib.save(cifti, map_output)
                                print(f"  Saved CIFTI map: {map_output}")

print("\n\n ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!")

LOADING RAW DATA (Running Once)
Loading Glasser parcellation from separate hemisphere files...

Left hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii
Right hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii

Left hemisphere: 180 parcels
  Parcel IDs range: 181 to 360
  Total vertices: 32492

Right hemisphere: 180 parcels
  Parcel IDs range: 1 to 180
  Total vertices: 32492
Loading Hemispheres...
Data Loaded. Left Valid: 29696, Right Valid: 29716

[FMRI] Preprocessing: Bin=1.0s, Delay=0s, HRF=True
  -> RSA: pe-av-small-16-frame [joint] | normalized_hrf_blockdiag
GLASSER PARCELLATION RSA PIPELINE [BLOCK-DIAGONAL (Within-Video)]

STEP 1: Computing RSA correlation map (left)...
----------------------------------------------------------------------
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Normalization (Z-score)...

--- STRATEGY: 

pixdim[1,2,3] should be non-zero; setting 0 dims to 1



RSA computation complete! Computed: 180, Skipped: 0, Failed: 0

PIPELINE COMPLETE!
Non-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/normalized_hrf_blockdiag/1s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_glasser_normalized_hrf_blockdiag_spearman_1s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | normalized_hrf_global
GLASSER PARCELLATION RSA PIPELINE [GLOBAL]

STEP 1: Computing RSA correlation map (left)...
----------------------------------------------------------------------
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Normalization (Z-score)...

--- STRATEGY: GLOBAL RSA ---
Final Model Vector elements: 4997541

Computing RSA for 180 parcels...
Progress: 50/180
Progress: 100/180
Progress: 150/180

RSA computation complete!

pixdim[1,2,3] should be non-zero; setting 0 dims to 1



RSA computation complete! Computed: 180, Skipped: 0, Failed: 0

PIPELINE COMPLETE!
Non-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/normalized_hrf_global/1s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_glasser_normalized_hrf_global_spearman_1s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | notnormalized_hrf_blockdiag
GLASSER PARCELLATION RSA PIPELINE [BLOCK-DIAGONAL (Within-Video)]

STEP 1: Computing RSA correlation map (left)...
----------------------------------------------------------------------
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Demeaning (Mean centering)...

--- STRATEGY: BLOCK-DIAGONAL (WITHIN-VIDEO) RSA ---
Final Model Vector elements: 316281

Computing RSA for 180 parcels...
Progress: 50/180
Progress: 100/

...


RSA computation complete! Computed: 180, Skipped: 0, Failed: 0

PIPELINE COMPLETE!
Non-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/normalized_global_delay5s/10s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_glasser_normalized_global_delay5s_spearman_10s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | notnormalized_blockdiag_delay5s
GLASSER PARCELLATION RSA PIPELINE [BLOCK-DIAGONAL (Within-Video)]

STEP 1: Computing RSA correlation map (left)...
----------------------------------------------------------------------
fMRI data shape: 32492 vertices × 311 timepoints
Performing Per-Run Demeaning (Mean centering)...

--- STRATEGY: BLOCK-DIAGONAL (WITHIN-VIDEO) RSA ---
Final Model Vector elements: 2921

Computing RSA for 180 parcels...
Progress: 50/180
Pro

pixdim[1,2,3] should be non-zero; setting 0 dims to 1



RSA computation complete! Computed: 180, Skipped: 0, Failed: 0

PIPELINE COMPLETE!
Non-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/notnormalized_blockdiag_delay5s/10s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_glasser_notnormalized_blockdiag_delay5s_spearman_10s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | notnormalized_global_delay5s
GLASSER PARCELLATION RSA PIPELINE [GLOBAL]

STEP 1: Computing RSA correlation map (left)...
----------------------------------------------------------------------
fMRI data shape: 32492 vertices × 311 timepoints
Performing Per-Run Demeaning (Mean centering)...

--- STRATEGY: GLOBAL RSA ---
Final Model Vector elements: 48205

Computing RSA for 180 parcels...
Progress: 50/180
Progress: 100/180
Progress: 150/180

RS

pixdim[1,2,3] should be non-zero; setting 0 dims to 1



RSA computation complete! Computed: 180, Skipped: 0, Failed: 0

PIPELINE COMPLETE!
Non-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/notnormalized_global_delay5s/10s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_glasser_notnormalized_global_delay5s_spearman_10s.dscalar.nii


 ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!


# summaries

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from glob import glob
import re

def calculate_top_10_percent(data):
    if len(data) == 0: return 0
    # Sort descending, take top 10%, return mean
    n_top = max(1, int(len(data) * 0.1))
    top_values = np.sort(data)[::-1][:n_top]
    return np.mean(top_values)

def extract_metadata(file_path):
    path_str = file_path.lower()
    parts = file_path.split(os.sep)
    
    # 1. Locate anchor to find Model and Design
    try:
        idx = parts.index('glasser_rsa_output')
        model_name = parts[idx + 1] # e.g., pe-av-small-16-frame
        design = parts[idx + 2]     # e.g., notnormalized_nohrf_blockdiag_delay6s
    except (ValueError, IndexError):
        model_name = "Unknown"
        design = "Unknown"

    # 2. Normalization
    normalization = "Normalized" if "notnormalized" not in design.lower() else "Not Normalized"
    
    # 3. HRF status
    # If "nohrf" or "nothrf" is in the design string -> No. Else -> Yes.
    has_no_hrf = "nohrf" in design.lower() or "nothrf" in design.lower()
    hrf_status = "No" if has_no_hrf else "Yes"
    
    # 4. Delay (Only if HRF is No)
    if hrf_status == "Yes":
        delay = "N/A"
    else:
        # Look for 'delay4s', 'delay5s', etc. inside the design folder name
        delay_match = re.search(r'delay(\d+s?)', design.lower())
        delay = delay_match.group(1) if delay_match else "N/A"

    # 5. Segment Length / Bin (Look for folder names like /1s/, /10s/)
    seg_match = re.search(r'/(\d+s)/', path_str)
    segment_len = seg_match.group(1) if seg_match else "N/A"
    
    # 6. Modality
    modality = "unknown"
    for m in ['joint', 'audio', 'video']:
        if f'/{m}/' in path_str:
            modality = m
            break
            
    # 7. Hemisphere
    hemi = "L" if "left" in path_str else "R"
    
    return model_name, design, hrf_status, delay, segment_len, normalization, modality, hemi

def generate_ranked_reports(root_dir):
    all_results = []
    
    # Recursive search
    files = glob(os.path.join(root_dir, "**/*.func.gii"), recursive=True)

    for f in files:
        # Exclusion filters
        if any(x in f.lower() for x in ["all func", "all_funcs", "projection-residual", "ols-residual", "concat"]):
            continue
            
        try:
            model, design, hrf, delay, seg, norm, mod, hemi = extract_metadata(f)
            
            if mod == "unknown": continue

            # Load GIFTI
            img = nib.load(f)
            data = img.darrays[0].data
            valid_data = data[~np.isnan(data)]
            
            if len(valid_data) > 0:
                all_results.append({
                    "Model": model,
                    "Design": design,
                    "HRF": hrf,
                    "Delay": delay,
                    "Segment": seg,
                    "Normalization": norm,
                    "Hemi": hemi,
                    "Avg_RSA": np.mean(valid_data),
                    "Top 10 Percent": calculate_top_10_percent(valid_data),
                    "Modality": mod 
                })
        except Exception:
            continue

    if not all_results:
        print("No valid RSA files found.")
        return

    full_df = pd.DataFrame(all_results)
    
    # Define Column Order
    display_cols = ["Model", "Design", "HRF", "Delay", "Segment", "Normalization", "Hemi", "Avg_RSA", "Top 10 Percent"]

    # Generate 3 Separate Tables
    for mod in ['joint', 'audio', 'video']:
        print("\n" + "="*140)
        print(f"MODALITY REPORT: {mod.upper()} (Sorted by Top 10%)")
        print("="*140)
        
        # Filter by modality
        mod_df = full_df[full_df['Modality'] == mod].copy()
        
        if mod_df.empty:
            print(f"No data found for {mod}")
        else:
            # Sort Descending by Top 10 Percent
            mod_df = mod_df.sort_values(by="Top 10 Percent", ascending=False)
            
            # Print
            print(mod_df[display_cols].to_string(index=False))

# --- EXECUTE ---
base_path = "/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame"
generate_ranked_reports(base_path)


MODALITY REPORT: JOINT (Sorted by Top 10%)
               Model                                Design HRF Delay Segment  Normalization Hemi  Avg_RSA  Top 10 Percent
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay5s  No    5s      5s     Normalized    L 0.167124        0.262399
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay5s  No    5s      5s     Normalized    R 0.159580        0.254421
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay6s  No    6s      5s     Normalized    L 0.159126        0.253614
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay5s  No    5s      2s     Normalized    L 0.164685        0.251870
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay5s  No    5s      2s     Normalized    R 0.159457        0.247452
pe-av-small-16-frame           notnormalized_hrf_blockdiag Yes   N/A      2s Not Normalized    L 0.159319        0.242627
pe-av-small-16-frame    normalized_nohrf_blockdiag_delay6s  No    6s      5s     Normalized    R 0.151

In [ ]:
import pandas as pd
import nibabel as nib
import numpy as np
import re
import os
from pathlib import Path

# ==========================================
# 1. SETUP PATHS
# ==========================================

RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")
LEFT_LABEL_PATH = '/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii'
RIGHT_LABEL_PATH = '/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii'

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_glasser_mapping(label_file_path):
    """Loads Glasser label file and returns {Key: Name} map and vertex array."""
    if not os.path.exists(label_file_path):
        raise FileNotFoundError(f"Label file not found: {label_file_path}")
    gii = nib.load(label_file_path)
    label_table = gii.labeltable.get_labels_as_dict()
    darray = gii.darrays[0]
    vertex_data = darray.data
    return label_table, vertex_data

def parse_file_info(filepath, base_dir):
    """
    Extracts analysis parameters AND Model Name from the file path.
    """
    path_obj = Path(filepath)
    
    # Calculate path relative to the base output directory to find Model Name
    try:
        rel_path = path_obj.relative_to(base_dir)
        # Structure: Model / Config / Bin / Modality / Hemisphere / File
        # rel_path.parts[0] is typically the Model Name
        model_name = rel_path.parts[0]
    except ValueError:
        model_name = "UnknownModel"

    full_path_str = str(filepath)
    meta = {
        'Filepath': str(filepath),
        'Model': model_name
    }
    
    # 1. Hemisphere
    if 'left_hemisphere' in full_path_str: meta['Hemisphere'] = 'Left'
    elif 'right_hemisphere' in full_path_str: meta['Hemisphere'] = 'Right'
    else: return None

    # 2. Modality
    if '/audio/' in full_path_str: meta['Modality'] = 'audio'
    elif '/video/' in full_path_str: meta['Modality'] = 'video'
    elif '/joint/' in full_path_str: meta['Modality'] = 'joint'
    else: meta['Modality'] = 'unknown'
    
    # 3. Bin Size
    bin_match = re.search(r'/(\d+)s/', full_path_str)
    meta['Bin_sec'] = int(bin_match.group(1)) if bin_match else 0
    
    # 4. Delay
    # If folder contains "nohrf", delay is usually implicitly 0 or irrelevant
    delay_match = re.search(r'delay(\d+)s', full_path_str)
    meta['Delay_sec'] = int(delay_match.group(1)) if delay_match else 0
    
    # 5. Normalization
    if 'notnormalized' in full_path_str: meta['Normalization'] = 'No'
    else: meta['Normalization'] = 'Yes'
            
    # 6. HRF
    if 'nohrf' in full_path_str: meta['HRF'] = 'No'
    else: meta['HRF'] = 'Yes'
            
    # 7. Strategy
    if 'blockdiag' in full_path_str: meta['Strategy'] = 'BlockDiag'
    else: meta['Strategy'] = 'Global'
    
    return meta

# ==========================================
# 3. MAIN PROCESSING
# ==========================================

print("Loading Parcellations...")
L_labels, L_vertices = get_glasser_mapping(LEFT_LABEL_PATH)
R_labels, R_vertices = get_glasser_mapping(RIGHT_LABEL_PATH)
print("Labels loaded.")

print(f"Crawling {RSA_RESULTS_BASE}...")
all_files = list(RSA_RESULTS_BASE.rglob("*.func.gii"))
print(f"Found {len(all_files)} files.")

data_rows = []

for i, fpath in enumerate(all_files):
    if i % 100 == 0: print(f"  {i}/{len(all_files)}...")
    
    # 1. Parse Metadata
    meta = parse_file_info(fpath, RSA_RESULTS_BASE)
    if not meta: continue
    
    # 2. Select Labels
    if meta['Hemisphere'] == 'Left':
        full_vertex_map = L_vertices
        labels_dict = L_labels
    else:
        full_vertex_map = R_vertices
        labels_dict = R_labels
        
    # 3. Load & Handle Vertex Mismatch
    try:
        img = nib.load(fpath)
        data_array = img.darrays[0].data
        
        n_data = data_array.shape[0]
        n_labels = full_vertex_map.shape[0]
        
        target_vertex_map = None
        
        # LOGIC TO HANDLE 32k vs 29k MISMATCH
        if n_data == n_labels:
            target_vertex_map = full_vertex_map
        else:
            # Check if it matches the "Dense" surface (medial wall removed)
            valid_mask = (full_vertex_map != 0) # Assumes 0 is medial wall
            n_valid = np.sum(valid_mask)
            
            if n_data == n_valid:
                # File is dense, so we must slice our label map to match it
                target_vertex_map = full_vertex_map[valid_mask]
            else:
                # Dimension is totally weird (e.g. 91k), skip
                # print(f"Skip {fpath.name}: Shape {n_data} mismatch.")
                continue

        # 4. Aggregate
        for label_key, label_name in labels_dict.items():
            if label_key == 0: continue
            
            # Mask against the ALIGNED vertex map
            mask = (target_vertex_map == label_key)
            
            if np.any(mask):
                val = np.mean(data_array[mask])
                row = {
                    'Parcel_Name': label_name,
                    'Label_Key': label_key,
                    'RSA_Correlation': val
                }
                row.update(meta)
                data_rows.append(row)
                
    except Exception as e:
        print(f"Error reading {fpath.name}: {e}")

# ==========================================
# 4. PIVOT
# ==========================================
if data_rows:
    df = pd.DataFrame(data_rows)
    
    # Construct Unique Condition Key (Includes MODEL now)
    df['Condition'] = (
        df['Model'] + '_' + 
        df['Modality'] + '_' +
        df['Normalization'].apply(lambda x: 'Norm' if x=='Yes' else 'NoNorm') + '_' +
        df['HRF'].apply(lambda x: 'HRF' if x=='Yes' else 'NoHRF') + '_' +
        df['Strategy'] + '_' +
        'delay' + df['Delay_sec'].astype(str) + '_' +
        'bin' + df['Bin_sec'].astype(str)
    )
    
    print("\nPivoting...")
    # Check for duplicates before pivoting to be safe
    duplicates = df[df.duplicated(subset=['Parcel_Name', 'Hemisphere', 'Label_Key', 'Condition'], keep=False)]
    if not duplicates.empty:
        print("WARNING: Duplicates found! Checking first few:")
        print(duplicates[['Model', 'Condition', 'Parcel_Name']].head())
        # Drop duplicates if they exist (taking the last one usually implies most recent overwrite)
        df = df.drop_duplicates(subset=['Parcel_Name', 'Hemisphere', 'Label_Key', 'Condition'], keep='last')

    df_wide = df.pivot(
        index=['Parcel_Name', 'Hemisphere', 'Label_Key'],
        columns='Condition',
        values='RSA_Correlation'
    )
    
    df_wide.reset_index(inplace=True)
    print(f"Success! Final Shape: {df_wide.shape}")
    
    df_wide.to_csv('/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame/glasser_rsa_wide_robust.csv', index=False)
    print("Saved to glasser_rsa_wide_robust.csv")
else:
    print("No data found.")

Loading Parcellations...
Labels loaded.
Crawling /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output...
Found 786 files.
  0/786...
  100/786...
  200/786...
  300/786...
  400/786...
  500/786...
  600/786...
  700/786...

Pivoting...
                      Model  \
42120  pe-av-small-16-frame   
42121  pe-av-small-16-frame   
42122  pe-av-small-16-frame   
42123  pe-av-small-16-frame   
42124  pe-av-small-16-frame   

                                               Condition Parcel_Name  
42120  pe-av-small-16-frame_audio_Norm_NoHRF_BlockDia...    L_V1_ROI  
42121  pe-av-small-16-frame_audio_Norm_NoHRF_BlockDia...   L_MST_ROI  
42122  pe-av-small-16-frame_audio_Norm_NoHRF_BlockDia...    L_V6_ROI  
42123  pe-av-small-16-frame_audio_Norm_NoHRF_BlockDia...    L_V2_ROI  
42124  pe-av-small-16-frame_audio_Norm_NoHRF_BlockDia...    L_V3_ROI  
Success! Final Shape: (360, 217)
Saved to glasser_rsa_wide_robust.csv


# Misc

In [1]:
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)
# Input fMRI files
mat_file_left = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
mat_file_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left =  "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"
# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 0.0
# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T
#remove the medial wall
row_std_left = fmri_left_raw.std(axis=1)
valid_mask_left = row_std_left > 0

print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_left)}")
fmri_left_processed = preprocess_fmri(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC)
fmri_left_processed[valid_mask_left] = (
    fmri_left_processed[valid_mask_left]- fmri_left_processed[valid_mask_left].mean(axis=1, keepdims=True)) / fmri_left_processed[valid_mask_left].std(axis=1, keepdims=True)
print(f"Processed left fMRI shape: {fmri_left_processed.shape}")
# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T
#remove the medial wall
row_std_right = fmri_right_raw.std(axis=1)
valid_mask_right = row_std_right > 0
print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_right)}")
fmri_right_processed = preprocess_fmri(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
fmri_right_processed[valid_mask_right] = (
    fmri_right_processed[valid_mask_right]- fmri_right_processed[valid_mask_right].mean(axis=1, keepdims=True)) / fmri_right_processed[valid_mask_right].std(axis=1, keepdims=True)
print(f"Processed right fMRI shape: {fmri_right_processed.shape}")

STEP 0: Processing fMRI with timing/segmentation


NameError: name 'load_glasser_parcellation' is not defined

### PE-AV

In [ ]:
# ============================================================================
# AUTOMATED MULTI-MODEL RSA LOOP, BLOCK DIAGONAL
# ============================================================================

# 1. Define Model Folders and their clean labels for filenames
# key: folder name on disk, value: label to use in the filename
preprocessing_label = 'normalized_hrf_blockdiag'
normalization=True
hrf=True
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")
models = {
    # "pe-av-base": "pe-av-base",
    # "pe-av-base-16-frame": "pe-av-base-16-frame",
    # "pe-av-large": "pe-av-large",
    # "pe-av-large-16-frame": "pe-av-large-16-frame",
    # "pe-av-small": "pe-av-small",
    "pe-av-small-16-frame": "pe-av-small-16-frame"
}

# 2. Define Modalities and their corresponding .npy suffixes
modalities = {
    "joint": "av",
    "audio": "a",
    "video": "v"
}

# Base Paths and configurations (Assuming boilerplate Step 0 has already loaded fMRI/Labels)
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")

for folder_name, model_label in models.items():
    for mod_name, mod_suffix in modalities.items():
        
        # Construct the specific label for this run
        current_run_label = f"{model_label}_{mod_name}"
        
        # Path to the specific .npy embedding file
        # Note: we use folder_name for the path, but model_label for the file prefix
        # Mapping 'pe-av-small-16-frame' -> 'peav_small_16frame' for the file prefix
        file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
        model_data = EMBEDDINGS_BASE / folder_name / "1s" /f"{file_prefix}_{mod_suffix}.npy"
        
        # Define and create output directory
        # e.g., .../glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/joint
        output_dir = RSA_RESULTS_BASE / f"{model_label}_{preprocessing_label}" /f"{int(BIN_SEC)}s" / mod_name
        
        if not model_data.exists():
            print(f"SKIPPING: File not found -> {model_data}")
            continue

        print(f"\n\n STARTING GLasser RSA: {model_label} [{mod_name.upper()}]")
        print(f"Using model file: {model_data}")

        # --- Process LEFT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='left',
            label=current_run_label,
            parcel_labels=left_labels,
            parcel_info=left_info,
            fmri_data=fmri_left_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "left_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=True
        )

        # --- Process RIGHT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='right',
            label=current_run_label,
            parcel_labels=right_labels,
            parcel_info=right_info,
            fmri_data=fmri_right_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "right_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=True
        )

print("\n\n ALL MODELS AND MODALITIES PROCESSED SUCCESSFULLY!")

## 1s bin normalized_hrf peav small 16 frame

In [ ]:
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)
# Input fMRI files
mat_file_left = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
mat_file_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left =  "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"
# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 1.0
DELAY_SEC = 0.0
# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T
#remove the medial wall
row_std_left = fmri_left_raw.std(axis=1)
valid_mask_left = row_std_left > 0

print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_left)}")
fmri_left_processed = preprocess_fmri(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC)
fmri_left_processed[valid_mask_left] = (
    fmri_left_processed[valid_mask_left]- fmri_left_processed[valid_mask_left].mean(axis=1, keepdims=True)) / fmri_left_processed[valid_mask_left].std(axis=1, keepdims=True)
print(f"Processed left fMRI shape: {fmri_left_processed.shape}")
# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T
#remove the medial wall
row_std_right = fmri_right_raw.std(axis=1)
valid_mask_right = row_std_right > 0
print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_right)}")
fmri_right_processed = preprocess_fmri(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
fmri_right_processed[valid_mask_right] = (
    fmri_right_processed[valid_mask_right]- fmri_right_processed[valid_mask_right].mean(axis=1, keepdims=True)) / fmri_right_processed[valid_mask_right].std(axis=1, keepdims=True)
print(f"Processed right fMRI shape: {fmri_right_processed.shape}")

In [ ]:
# ============================================================================
# AUTOMATED MULTI-MODEL RSA LOOP, BLOCK DIAGONAL
# ============================================================================

# 1. Define Model Folders and their clean labels for filenames
# key: folder name on disk, value: label to use in the filename
preprocessing_label = 'normalized_hrf_blockdiag'
normalization=True
hrf=True
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")
models = {
    # "pe-av-base": "pe-av-base",
    # "pe-av-base-16-frame": "pe-av-base-16-frame",
    # "pe-av-large": "pe-av-large",
    # "pe-av-large-16-frame": "pe-av-large-16-frame",
    # "pe-av-small": "pe-av-small",
    "pe-av-small-16-frame": "pe-av-small-16-frame"
}

# 2. Define Modalities and their corresponding .npy suffixes
modalities = {
    "joint": "av",
    "audio": "a",
    "video": "v"
}

# Base Paths and configurations (Assuming boilerplate Step 0 has already loaded fMRI/Labels)
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")

for folder_name, model_label in models.items():
    for mod_name, mod_suffix in modalities.items():
        
        # Construct the specific label for this run
        current_run_label = f"{model_label}_{mod_name}"
        
        # Path to the specific .npy embedding file
        # Note: we use folder_name for the path, but model_label for the file prefix
        # Mapping 'pe-av-small-16-frame' -> 'peav_small_16frame' for the file prefix
        file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
        model_data = EMBEDDINGS_BASE / folder_name / "1s" /f"{file_prefix}_{mod_suffix}.npy"
        
        # Define and create output directory
        # e.g., .../glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/joint
        output_dir = RSA_RESULTS_BASE / f"{model_label}_{preprocessing_label}" /f"{int(BIN_SEC)}s" / mod_name
        
        if not model_data.exists():
            print(f"SKIPPING: File not found -> {model_data}")
            continue

        print(f"\n\n STARTING GLasser RSA: {model_label} [{mod_name.upper()}]")
        print(f"Using model file: {model_data}")

        # --- Process LEFT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='left',
            label=current_run_label,
            parcel_labels=left_labels,
            parcel_info=left_info,
            fmri_data=fmri_left_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "left_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=True
        )

        # --- Process RIGHT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='right',
            label=current_run_label,
            parcel_labels=right_labels,
            parcel_info=right_info,
            fmri_data=fmri_right_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "right_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=True
        )

print("\n\n ALL MODELS AND MODALITIES PROCESSED SUCCESSFULLY!")

In [ ]:
# ============================================================================
# AUTOMATED MULTI-MODEL RSA LOOP, GLOBAL
# ============================================================================

# 1. Define Model Folders and their clean labels for filenames
# key: folder name on disk, value: label to use in the filename
preprocessing_label = 'normalized_hrf_global'
normalization=True
hrf=True
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")
models = {
    # "pe-av-base": "pe-av-base",
    # "pe-av-base-16-frame": "pe-av-base-16-frame",
    # "pe-av-large": "pe-av-large",
    # "pe-av-large-16-frame": "pe-av-large-16-frame",
    # "pe-av-small": "pe-av-small",
    "pe-av-small-16-frame": "pe-av-small-16-frame"
}

# 2. Define Modalities and their corresponding .npy suffixes
modalities = {
    "joint": "av",
    "audio": "a",
    "video": "v"
}

# Base Paths and configurations (Assuming boilerplate Step 0 has already loaded fMRI/Labels)
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")

for folder_name, model_label in models.items():
    for mod_name, mod_suffix in modalities.items():
        
        # Construct the specific label for this run
        current_run_label = f"{model_label}_{mod_name}"
        
        # Path to the specific .npy embedding file
        # Note: we use folder_name for the path, but model_label for the file prefix
        # Mapping 'pe-av-small-16-frame' -> 'peav_small_16frame' for the file prefix
        file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
        model_data = EMBEDDINGS_BASE / folder_name / "1s" /f"{file_prefix}_{mod_suffix}.npy"
        
        # Define and create output directory
        # e.g., .../glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/joint
        output_dir = RSA_RESULTS_BASE / f"{model_label}_{preprocessing_label}" /f"{int(BIN_SEC)}s" / mod_name
        
        if not model_data.exists():
            print(f"SKIPPING: File not found -> {model_data}")
            continue

        print(f"\n\n STARTING GLasser RSA: {model_label} [{mod_name.upper()}]")
        print(f"Using model file: {model_data}")

        # --- Process LEFT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='left',
            label=current_run_label,
            parcel_labels=left_labels,
            parcel_info=left_info,
            fmri_data=fmri_left_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "left_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=False
        )

        # --- Process RIGHT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='right',
            label=current_run_label,
            parcel_labels=right_labels,
            parcel_info=right_info,
            fmri_data=fmri_right_processed,
            timing_df=timing_df,bin_sec=BIN_SEC,
            model_data=str(model_data),
            output_dir=str(output_dir / "right_hemisphere"),
            normalization=normalization, hrf=hrf,
            method=RSA_METHOD, block_diagonal=False
        )

print("\n\n ALL MODELS AND MODALITIES PROCESSED SUCCESSFULLY!")

### notnormalized_nohrf

In [ ]:
# ============================================================================
# AUTOMATED MULTI-MODEL RSA LOOP
# ============================================================================

# 1. Define Model Folders and their clean labels for filenames
# key: folder name on disk, value: label to use in the filename
preprocessing_label = 'notnormalized_nohrf'
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")
models = {
    "pe-av-base": "pe-av-base",
    "pe-av-base-16-frame": "pe-av-base-16-frame",
    "pe-av-large": "pe-av-large",
    "pe-av-large-16-frame": "pe-av-large-16-frame",
    "pe-av-small": "pe-av-small",
    "pe-av-small-16-frame": "pe-av-small-16-frame"
}

# 2. Define Modalities and their corresponding .npy suffixes
modalities = {
    "joint": "av",
    "audio": "a",
    "video": "v"
}

# Base Paths and configurations (Assuming boilerplate Step 0 has already loaded fMRI/Labels)
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")

for folder_name, model_label in models.items():
    for mod_name, mod_suffix in modalities.items():
        
        # Construct the specific label for this run
        current_run_label = f"{model_label}_{mod_name}"
        
        # Path to the specific .npy embedding file
        # Note: we use folder_name for the path, but model_label for the file prefix
        # Mapping 'pe-av-small-16-frame' -> 'peav_small_16frame' for the file prefix
        file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', 'frame')
        model_data = EMBEDDINGS_BASE / folder_name / f"{file_prefix}_{mod_suffix}.npy"
        
        # Define and create output directory
        # e.g., .../glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/joint
        output_dir = RSA_RESULTS_BASE / f"{model_label}_{preprocessing_label}_bin{BIN_SEC}s" / mod_name
        
        if not model_data.exists():
            print(f"SKIPPING: File not found -> {model_data}")
            continue

        print(f"\n\n STARTING GLasser RSA: {model_label} [{mod_name.upper()}]")
        print(f"Using model file: {model_data}")

        # --- Process LEFT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='left',
            label=current_run_label,
            parcel_labels=left_labels,
            parcel_info=left_info,
            fmri_data=fmri_left_processed,timing_df=timing_df,
            model_data=str(model_data),
            normalization=False, hrf=False,
            output_dir=str(output_dir / "left_hemisphere"),
            method=RSA_METHOD
        )

        # --- Process RIGHT Hemisphere ---
        complete_rsa_pipeline(
            hemisphere='right',
            label=current_run_label,
            parcel_labels=right_labels,
            parcel_info=right_info,
            fmri_data=fmri_right_processed,
            timing_df=timing_df,
            model_data=str(model_data),
            normalization=False, hrf=False,
            output_dir=str(output_dir / "right_hemisphere"),
            method=RSA_METHOD
        )

print("\n\n ALL MODELS AND MODALITIES PROCESSED SUCCESSFULLY!")

## LOO with PE-AV-small-16-frame audiovisual embeddings

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from pathlib import Path
import os
import sys

# ==========================================
# 1. CONFIGURATION & VARIABLES
# ==========================================

# File Paths
mat_file_left = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
mat_file_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"
glasser_label_left =  "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"

MODEL_EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")

# Model Specifics
# ------------------------------------------
model_label = "pe-av-small-16-frame"  # Used for folder naming
folder_name = "pe-av-small-16-frame"  # Used for input path
mod_name = "joint"                    # Modality name for output
mod_suffix = "av"                     # Suffix in .npy file (e.g., _av.npy)

# Construct Model Path
# Expecting: .../pe-av-small-16-frame/peav_small_16frame_av.npy
file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', 'frame')
model_file_path = MODEL_EMBEDDINGS_BASE / folder_name / f"{file_prefix}_{mod_suffix}.npy"

# RSA Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 4.0

# ==========================================
# 2. DATA LOADING
# ==========================================

print("=" * 70)
print("STEP 1: Loading Data")
print("=" * 70)

# Load Glasser Parcellation
# (Assumes load_glasser_parcellation is defined in your environment)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load fMRI - LEFT
print("\n--- Loading LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T
print(f"Raw Left Shape: {fmri_left_raw.shape}")

# Load fMRI - RIGHT
print("\n--- Loading RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T
print(f"Raw Right Shape: {fmri_right_raw.shape}")

# Load Model
print(f"\n--- Loading Model: {model_label} ---")
if not model_file_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_file_path}")

full_model_data = np.load(model_file_path)
full_model_data = (full_model_data - full_model_data.mean(axis=0, keepdims=True)) / full_model_data.std(axis=0, keepdims=True)
print(f"Model Data Shape: {full_model_data.shape}")

# ==========================================
# 3. PRE-PROCESSING (Segmenting fMRI)
# ==========================================

print("\n" + "="*70)
print("STEP 2: load the preprocessed fMRI")
print("=" * 70)

# Verify Dimensions
n_vertices, n_timepoints = fmri_left_processed.shape
n_model_time, n_features = full_model_data.shape

if n_timepoints != n_model_time:
    raise ValueError(f"Mismatch! fMRI time ({n_timepoints}) != Model time ({n_model_time})")

# ==========================================
# 4. LEAVE-ONE-OUT RSA LOOP
# ==========================================

# Define Output Directory for Video LOO
# .../glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/video
loo_base_dir = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output" / f"{model_label}_hrf_bin2s/joint/loo")
os.makedirs(loo_base_dir, exist_ok=True)

# Calculate Video Indices
video_bins = (timing_df['duration_sec'] / BIN_SEC).astype(int).values
video_indices = []
current_idx = 0
for b in video_bins:
    video_indices.append((current_idx, current_idx + b))
    current_idx += b

print("\n" + "="*70)
print(f"STEP 3: STARTING LEAVE-ONE-OUT LOOP (1 to 18)")
print(f"Output Dir: {loo_base_dir}")
print("="*70)

for i in range(18):
    vid_num = i + 1
    omit_vid_name = f"video{vid_num}"
    
    # A. Create Mask (True = Keep, False = Omit)
    omit_start, omit_end = video_indices[i]
    keep_mask = np.ones(n_timepoints, dtype=bool)
    keep_mask[omit_start:omit_end] = False
    
    print(f"\n--- Iteration {vid_num}/18: Omitting {omit_vid_name} ---")
    
    # B. Slice Data
    # IMPORTANT: fMRI is (Vertices, Time) -> Slice axis 1
    fmri_left_subset = fmri_left_processed[:, keep_mask]
    fmri_right_subset = fmri_right_processed[:, keep_mask]
    
    # IMPORTANT: Model is (Time, Features) -> Slice axis 0
    model_subset = full_model_data[keep_mask, :]
    
    # C. Save Temp Model
    temp_model_filename = f"temp_model_omit_{vid_num}.npy"
    temp_model_path = loo_base_dir / temp_model_filename
    np.save(temp_model_path, model_subset)
    
    # D. Define Label
    # We want: ..._omit_{vidn}.func.gii
    # We construct the label so the pipeline creates: rsa_{label}_{hemi}.func.gii
    # Final desired: rsa_pe-av-small-16-frame_joint_glasser_spearman_{hemi}_omit_{vidn}.func.gii
    
    # Note: If your 'complete_rsa_pipeline' automatically appends '_glasser_spearman',
    # we just need the prefix. If it doesn't, we add it. 
    # Based on standard logic, let's use:
    loo_run_label = f"{file_prefix}_{mod_name}_omit_{vid_num}"

    # E. Run Pipeline
    
    # Left
    complete_rsa_pipeline(
        hemisphere='left',
        label=loo_run_label,
        parcel_labels=left_labels,
        parcel_info=left_info,
        fmri_data=fmri_left_subset,
        model_file=str(temp_model_path),
        output_dir=str(loo_base_dir / "left_hemisphere"),
        method=RSA_METHOD
    )
    
    # Right
    complete_rsa_pipeline(
        hemisphere='right',
        label=loo_run_label,
        parcel_labels=right_labels,
        parcel_info=right_info,
        fmri_data=fmri_right_subset,
        model_file=str(temp_model_path),
        output_dir=str(loo_base_dir / "right_hemisphere"),
        method=RSA_METHOD
    )
    
    # F. Cleanup
    if temp_model_path.exists():
        os.remove(temp_model_path)

print("\n" + "="*70)
print("LEAVE-ONE-OUT LOOP COMPLETED SUCCESSFULLY")
print("="*70)

## unimodal models

In [ ]:
# ============================================================================
# UNIMODAL EMBEDDINGS RSA LOOP
# ============================================================================

# List of unimodal embedding files as provided
unimodal_files = [
    "AudioMAE_embeddings.npy",
    "clap-large_embeddings.npy",
    "pecore-l14_embeddings.npy",
    "videomae-large_embeddings.npy",
    "wavlm-large_embeddings.npy",
    "whisper-large-v3_embeddings.npy"
]

# Base Configurations
EMBEDDINGS_DIR = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/unimodal")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output")

# Ensure the method matches your previous config
RSA_METHOD = 'spearman' 

for file_name in unimodal_files:
    # 1. Extract model ID (e.g., 'AudioMAE_embeddings.npy' -> 'AudioMAE')
    model_id = file_name.replace("_embeddings.npy", "")
    
    # 2. Define full path to the input model file
    model_file = EMBEDDINGS_DIR / file_name
    
    # 3. Define and create the output directory for this specific model
    # Pattern: /home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/AudioMAE
    iter_output_dir = RSA_RESULTS_BASE / model_id
    iter_output_dir.mkdir(parents=True, exist_ok=True)

    if not model_file.exists():
        print(f" Skipping: {model_file} not found.")
        continue

    print(f"\n{'='*80}")
    print(f"PROCESSING UNIMODAL MODEL: {model_id}")
    print(f"Input: {model_file}")
    print(f"{'='*80}")

    # 4. Execute for LEFT Hemisphere
    # Function will save: rsa_AudioMAE_glasser_spearman_left.func.gii
    print(f"\n--- Running LEFT: {model_id} ---")
    complete_rsa_pipeline(
        hemisphere='left',
        label=model_id,
        parcel_labels=left_labels,
        parcel_info=left_info,
        fmri_data=fmri_left_processed,
        model_file=str(model_file),
        output_dir=str(iter_output_dir),
        method=RSA_METHOD
    )

    # 5. Execute for RIGHT Hemisphere
    # Function will save: rsa_AudioMAE_glasser_spearman_right.func.gii
    print(f"\n--- Running RIGHT: {model_id} ---")
    complete_rsa_pipeline(
        hemisphere='right',
        label=model_id,
        parcel_labels=right_labels,
        parcel_info=right_info,
        fmri_data=fmri_right_processed,
        model_file=str(model_file),
        output_dir=str(iter_output_dir),
        method=RSA_METHOD
    )

print("\n\nALL UNIMODAL RSA ANALYSES COMPLETED!")

## cav-mae-sync

### audio

In [ ]:
#### Configuration
label = 'cavmaesync_audio'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cavmaesync_delay4s_bin2s/audio")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync/2s/embeddings_audio_averaged.npy")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(hemisphere = 'left',label= label, 
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(hemisphere = 'right',label= label, 
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

### video

In [ ]:
#### Configuration
label = 'cavmaesync_video'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cavmaesync_delay4s_bin2s/video")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Input fMRI files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = DATA_DIR / "Data/movie_timing.csv"

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync/2s/embeddings_video_averaged.npy")

# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 0.0

# ============================================================================
# STEP 0: Processing fMRI with timing/segmentation
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = preprocess_fmri(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = preprocess_fmri(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(hemisphere = 'left',label= label, 
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(hemisphere = 'right',label= label, 
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

### concat

In [ ]:
#### Configuration
label = 'cavmaesync_concat'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/cavmaesync_delay4s_bin2s/concat")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Input fMRI files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = DATA_DIR / "Data/movie_timing.csv"

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/cav-mae-sync/2s/embeddings_concat_averaged.npy")

# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 0.0

# ============================================================================
# STEP 0: Processing fMRI with timing/segmentation
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = preprocess_fmri(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = preprocess_fmri(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(hemisphere = 'left',label= label, 
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(hemisphere = 'right',label= label, 
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func_left.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func_right.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)